# GraphRAG

## Import packages

In [8]:
import sys
sys.path.append('..')
sys.path.append('../neurorag')
sys.path.append('../neurorag/chains')

import os
import time
import pandas as pd
from tqdm import tqdm
from pathlib import Path
import json
from dotenv import load_dotenv
from getpass import getpass

from neurorag.neurorag import NeuroRAG

from metrics import (
  embeddings_cosine_sim_metric,
  bleu_metric,
  rogue_l_metric,
  rogue_1_metric,
  factscore_metric,
  bert_score_metric,
)

## Disable warnings

In [9]:
import warnings
warnings.filterwarnings('ignore')

## Setup environment variables

You have to define the following environment variables in the `.env` file, terminal environment, or input field within this Jupyter notebook.

## Import packages

In [10]:
env_variables = [
  'TAVILY_API_KEY',
  'ENTREZ_EMAIL',
  'OPENROUTER_API_KEY',
  'CHROMA_API_KEY',
  'CHROMA_TENANT',
  'CHROMA_DATABASE',
  'CHROMA_COLLECTION_NAME',
]

load_dotenv()

for key in env_variables:
  value = os.getenv(key)

  if value is None:
    value = getpass(key)

  os.environ[key] = value

## Build model

In [11]:
app = NeuroRAG(
  debug=True,
  answer_style='Answer in 2-3 sentences (30-50 words). Use plain, accessible language like a medical textbook — avoid technical jargon and specific brain region names unless the question asks for them. Explain the concept clearly, as in a PubMed summary.',
)
app.compile()

## Evaluate RAG

### Load QA dataset

In [12]:
mediqa_df = pd.read_csv('../datasets/pubmed_summary_qa.csv')[:50]
mediqa_df

,question,answer
0,Which brain region is involved in working memo...,The dorsolateral prefrontal cortex (DLPFC) is ...
1,Are other brain regions also involved in worki...,"Yes, other brain regions, such as the premotor..."
2,What is a visuomotor task?,A visuomotor task is a type of task that requi...
3,What brain regions are involved in visuomotor ...,The brain regions involved in visuomotor trans...
4,What is the role of the prefrontal cortex in v...,The prefrontal cortex is involved in the prepa...
5,How does the auditory system respond to differ...,The auditory system's response to sound varies...
6,What is tonotopic organization in the auditory...,Tonotopic organization refers to the mapping o...
7,What brain regions are involved in language pr...,Language processing involves areas in the pref...
8,How does bilingualism affect language processi...,Bilingualism is associated with overlapping ac...
9,What is functional magnetic resonance imaging ...,Functional magnetic resonance imaging (fMRI) i...


### Load cached RAGs responses

In [13]:
cache_path = Path('cache.json')

if not os.path.exists(cache_path):
  data = {}
  with open(cache_path, 'w') as file:
    json.dump(data, file)

with open(cache_path, 'r') as f:
  cache = json.load(f)

CACHE_KEY = 'text-to-text-neurorag-evaluation'

if CACHE_KEY not in cache:
  cache[CACHE_KEY] = {}

len(cache[CACHE_KEY].keys())

17

In [14]:
questions = list(mediqa_df['question'].tolist())
expected_answers = list(mediqa_df['answer'].tolist())
predicted_answers = []
generation_times = []

for index, question in tqdm(enumerate(questions)):
  if question not in cache[CACHE_KEY]:
    start_time = time.perf_counter()
    cache[CACHE_KEY][question] = app.invoke(question)['generation']
    elapsed = time.perf_counter() - start_time
    generation_times.append(elapsed)

  predicted_answers.append(cache[CACHE_KEY][question])

  with open(cache_path, 'w') as f:
    json.dump(cache, f)

if generation_times:
  print(f'Generation times (n={len(generation_times)}):')
  print(f'  Mean:   {sum(generation_times) / len(generation_times):.2f}s')
  print(f'  Median: {sorted(generation_times)[len(generation_times) // 2]:.2f}s')
  print(f'  Min:    {min(generation_times):.2f}s')
  print(f'  Max:    {max(generation_times):.2f}s')
  print(f'  Total:  {sum(generation_times):.2f}s')
else:
  print('All answers loaded from cache, no generation times recorded.')

cos_score = embeddings_cosine_sim_metric(expected_answers, predicted_answers)
print('cos_score', cos_score)
bleu_score = bleu_metric(expected_answers, predicted_answers)
print('bleu_score', bleu_score)
rogue_1_score = rogue_1_metric(expected_answers, predicted_answers)
print('rogue_1_score', rogue_1_score)
rogue_l_score = rogue_l_metric(expected_answers, predicted_answers)
print('rogue_l_score', rogue_l_score)
factscore_score = factscore_metric(expected_answers, predicted_answers)
print('factscore_score', factscore_score)
bert_score = bert_score_metric(expected_answers, predicted_answers)
print('bert_score', bert_score)

0it [00:00, ?it/s]

[2026-03-31 23:01:54.143] ---GENERATE STEP-BACK QUERY---
[2026-03-31 23:01:55.782] ---GENERATE SUBQUERIES---
[2026-03-31 23:01:56.216] ---DETERMINE SPECIALIZED SOURCES---
[2026-03-31 23:01:56.705] ---SELECTED SOURCES: ['vectorstore', 'pubmed']---
[2026-03-31 23:01:56.705] ---ROUTE QUESTION---
[2026-03-31 23:01:56.706] ---GENERATE HYDE DOCUMENTS---
[2026-03-31 23:01:57.881][2026-03-31 23:01:57.881] ---RETRIEVE FROM VECTOR STORE---
 ---RETRIEVE FROM PUBMED---
[2026-03-31 23:01:58.460] pub_med_retriever_node HTTP Error 429: Too Many Requests
Too Many Requests, waiting for 0.20 seconds...
Too Many Requests, waiting for 0.20 seconds...
Too Many Requests, waiting for 0.80 seconds...
Too Many Requests, waiting for 1.60 seconds...
Too Many Requests, waiting for 1.60 seconds...
Too Many Requests, waiting for 6.40 seconds...
[2026-03-31 23:02:11.095] ---GRADE DOCUMENTS---
[2026-03-31 23:02:11.096] ---RRF RANKED: 20 documents---
[2026-03-31 23:02:11.100] ---BM25 CANDIDATES: 12 documents---
[2026-

18it [01:09,  3.83s/it]

[2026-03-31 23:03:03.079] ---GRADE GENERATION---
[2026-03-31 23:03:03.088] ---GENERATE STEP-BACK QUERY---
[2026-03-31 23:03:04.079] ---GENERATE SUBQUERIES---
[2026-03-31 23:03:04.403] ---DETERMINE SPECIALIZED SOURCES---
[2026-03-31 23:03:04.778] ---SELECTED SOURCES: ['vectorstore']---
[2026-03-31 23:03:04.778] ---ROUTE QUESTION---
[2026-03-31 23:03:04.778] ---GENERATE HYDE DOCUMENTS---
[2026-03-31 23:03:06.617] ---RETRIEVE FROM VECTOR STORE---
[2026-03-31 23:03:08.551] ---GRADE DOCUMENTS---
[2026-03-31 23:03:08.552] ---RRF RANKED: 9 documents---
[2026-03-31 23:03:08.553] ---BM25 CANDIDATES: 9 documents---
[2026-03-31 23:03:09.973] ---RERANKED TOP DOCS: 1 (scores: [5, 0, 0, 0, 0])---
[2026-03-31 23:03:10.495] ---FINAL DOCUMENTS: 0---
[2026-03-31 23:03:10.495] ---ASSESS GRADED DOCUMENTS---
[2026-03-31 23:03:10.495] ---DECISION: SOME DOCUMENTS ARE NOT RELEVANT TO QUESTION, INCLUDE WEB SEARCH---
[2026-03-31 23:03:10.496] ---WEB SEARCH---
[2026-03-31 23:03:15.607] ---GENERATE---
[2026-03-31

19it [01:52,  6.74s/it]

...

[2026-03-31 23:55:07.577] ---DECISION: GENERATION ADDRESSES QUESTION---
[2026-03-31 23:55:07.588] ---GENERATE STEP-BACK QUERY---
[2026-03-31 23:55:07.978] ---GENERATE SUBQUERIES---
[2026-03-31 23:55:09.109] ---DETERMINE SPECIALIZED SOURCES---
[2026-03-31 23:55:09.450] ---SELECTED SOURCES: ['vectorstore', 'pubmed']---
[2026-03-31 23:55:09.451] ---ROUTE QUESTION---
[2026-03-31 23:55:09.451] ---GENERATE HYDE DOCUMENTS---
[2026-03-31 23:55:12.290][2026-03-31 23:55:12.291] ---RETRIEVE FROM VECTOR STORE---
 ---RETRIEVE FROM PUBMED---
[2026-03-31 23:55:13.042] pub_med_retriever_node HTTP Error 429: Too Many Requests
Too Many Requests, waiting for 26843545.60 seconds...
Too Many Requests, waiting for 26843545.60 seconds...
[2026-03-31 23:57:12.294] pub_med_retriever_node timed out
[2026-03-31 23:57:12.294] ---GRADE DOCUMENTS---
[2026-03-31 23:57:12.294] ---RRF RANKED: 12 documents---
[2026-03-31 23:57:12.295] ---BM25 CANDIDATES: 12 documents---
[2026-03-31 23:57:16.700] ---RERANKED TOP DOCS: 5

50it [56:02, 67.26s/it] 

[2026-03-31 23:57:56.882] ---GRADE GENERATION---
Generation times (n=33):
  Mean:   93.18s
  Median: 67.29s
  Min:    16.44s
  Max:    175.94s
  Total:  3074.95s


cos_score 0.7694080629230321
bleu_score 0.025920952201597718
rogue_1_score 0.28967709800723684
rogue_l_score 0.2233857580065206
factscore_score 0.1956073031444439
bert_score 0.2182588428258896


Too Many Requests, waiting for 214748364.80 seconds...
Too Many Requests, waiting for 214748364.80 seconds...
Too Many Requests, waiting for 214748364.80 seconds...
